In [14]:
import cv2
import pandas as pd
import numpy as np

In [15]:
video_path = r"C:\UAV\UAV_SWARM\uav week3\balloons.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Video not loaded")
else:
    print("Video Loaded Successfully")

Video Loaded Successfully


In [16]:
ret, frame1 = cap.read()

if not ret:
    print("Cannot read video")

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

gray1 = cv2.GaussianBlur(gray1,(5,5),0)

In [17]:
while True:

    ret, frame2 = cap.read()

    if not ret:
        break


    # Convert current frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)


    # Gaussian Blur
    gray2 = cv2.GaussianBlur(gray2,(5,5),0)


    ###################################
    # Frame Difference
    ###################################

    difference = cv2.absdiff(gray1, gray2)


    ###################################
    # Threshold
    ###################################

    _, thresh = cv2.threshold(
        difference,
        30,
        255,
        cv2.THRESH_BINARY
    )


    ###################################
    # Morphological Operations
    ###################################

    kernel = np.ones((5,5),np.uint8)


    # Remove small noise
    opening = cv2.morphologyEx(
        thresh,
        cv2.MORPH_OPEN,
        kernel
    )


    # Fill holes
    closing = cv2.morphologyEx(
        opening,
        cv2.MORPH_CLOSE,
        kernel
    )


    ###################################
    # Contours
    ###################################

    contours,_ = cv2.findContours(
        closing,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )


    motion_frame = frame2.copy()


    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > 500:

            x,y,w,h = cv2.boundingRect(cnt)

            cv2.rectangle(
                motion_frame,
                (x,y),
                (x+w,y+h),
                (0,255,0),
                2
            )


            cv2.putText(
                motion_frame,
                "Motion Detected",
                (10,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,0,255),
                2
            )


    ###################################
    # Display
    ###################################

    cv2.imshow(
        "Original Video",
        frame2
    )

    cv2.imshow(
        "Frame Difference",
        difference
    )

    cv2.imshow(
        "Threshold",
        thresh
    )

    cv2.imshow(
        "Morphological Result",
        closing
    )

    cv2.imshow(
        "Motion Detection",
        motion_frame
    )


    gray1 = gray2.copy()


    if cv2.waitKey(30) & 0xff == ord('q'):
        break



cap.release()
cv2.destroyAllWindows()

In [18]:
df = pd.DataFrame(
    motion_log,
    columns=["Frame Number", "Motion Objects"]
)

df.to_csv("motion_log.csv", index=False)

print("CSV Saved Successfully")

CSV Saved Successfully


In [19]:
cap.release()
cv2.destroyAllWindows()